In [ ]:
import torch   
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.amp import autocast 




In [2]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

with_cuda = torch.cuda.is_available()
if with_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [3]:
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import os
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import random_split, DataLoader, Dataset

In [4]:
def normalization(x,mini , maxi):
    return (x - mini) / (maxi - mini)

def denormalization(x ,mini , maxi):
    return x * (maxi - mini) + mini

In [5]:
class SpectrogramDataset(Dataset):
    def __init__(self, spec_path):
        self.file_paths = [os.path.join(spec_path, f)
                           for f in os.listdir(spec_path) if f.endswith('.npy')]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        log_spectrogram = np.load(self.file_paths[idx])
        log_spectrogram_tensor = torch.from_numpy(log_spectrogram).float()
        mini, maxi = log_spectrogram_tensor.min(), log_spectrogram_tensor.max()
        normalize_tensor = (log_spectrogram_tensor - mini) / (maxi - mini + 1e-8)
        return normalize_tensor, mini, maxi

def get_loader(spec_path, batch_size=5):
    dataset = SpectrogramDataset(spec_path)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

def get_data(spec_path, batch_size= 12 , split_ratio=0.8):
    dataset = SpectrogramDataset(spec_path)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    # Split aléatoire du dataset
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    # Création des DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) #numworkers = 4
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader


In [ ]:
def testeur(data_loader, model, device , epoch , loss_test, loss_affichage, recon_loss_plot , kl_loss_plot , minim, maxim):
    path = "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P1\\projet deep L3S2\\tentative_generation"
    
    
    plt.figure(figsize=(10, 4))
    plt.plot(kl_loss_plot, label='KL Divergence Loss', color='orange')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('KL Divergence Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(path + "\\loss", f"loss_reconstruction_kl_{epoch}epochs.png"))
    plt.close()
    
    plt.figure(figsize=(10, 4))
    plt.plot(recon_loss_plot, label='Reconstruction Loss', color='blue')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('KL Divergence Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(path + "\\loss", f"loss_reconstruction_{epoch}epochs.png"))
    plt.close()
    
    
    original = data_loader.dataset[8]
    original = original.unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Reconstruction
        original_input = original.permute(0, 2, 1)  # (1, 2000, 1025)
        reconstructed, mu, logvar = model(original_input)  # 👈 changement ici
        reconstructed = reconstructed.permute(0, 2, 1) 
    
   

    original = original.squeeze(0).cpu().detach().numpy()
    reconstructed = reconstructed.squeeze(0).cpu().detach().numpy()


    original = denormalization(original, minim, maxim)

    reconstructed2 = denormalization(reconstructed, minim, maxim)
        
        
    def save_audio_original(path, spec, epoch):
        # Enregistrer le fichier audio
        audio_path = os.path.join(path + "\\son", f"audio_original.wav")
        spec_amp = librosa.db_to_amplitude(spec)
        signal = librosa.griffinlim(spec_amp, hop_length=512)
        sf.write(audio_path, signal, 22050)
         
    
    def save_audio_reconstructed(path, spec, epoch):
        # Enregistrer le fichier audio
        audio_path = os.path.join(path + "\\son", f"audio_reconstruit{epoch}.wav")
        spec_amp = librosa.db_to_amplitude(spec)
        signal = librosa.griffinlim(spec_amp, hop_length=512)

        sf.write(audio_path, signal, 22050)

    def save_image_original(path, image, epoch):
        # Enregistrer l'image
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(image, y_axis='log', x_axis='time')
        plt.colorbar(format='%+2.0f dB')
        plt.title(f"{image} : {epoch}")
        img_path = os.path.join(path + "\\images" , "images_original.png")
        plt.savefig(img_path)
        plt.close()

    def save_image_reconstruit(path, image, epoch):
        # Enregistrer l'image
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(image, y_axis='log', x_axis='time')
        plt.colorbar(format='%+2.0f dB')
        plt.title(f"{image} : {epoch}")
        img_path = os.path.join(path + "\\images", f"images_reconstuites{epoch}.png")
        plt.savefig(img_path)
        plt.close()
    
    save_audio_original(path, original, epoch)
    save_audio_reconstructed(path, reconstructed2, epoch)
    save_image_original(path, original, epoch)
    save_image_reconstruit(path, reconstructed2, epoch)
    
    if epoch > 0 : 
        plt.figure(figsize=(10, 4))
        train = list(range(len(loss_affichage)))         # [0, 1, 2, ..., 49]
        test = list(range(0, epoch+1 , 10))   # [0, 10, 20, 30, 40]
   
        # 2. Les courbes
        plt.plot(train, loss_affichage, label='Train Loss', color='blue')
        plt.plot(test, loss_test, label='Test Loss', color='orange', marker='o')

        # 3. Habillage
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.title('Train vs Test Loss')
        plt.legend()
        plt.grid(True)

        # 4. Sauvegarde
        plt.savefig(os.path.join(path + "\\loss", f"loss_comparaison_{epoch}epochs.png"))
        plt.close()

    
    if epoch > 30 : 
        loss__test_tronque = loss_test[1:]
        loss_affichage_tronque = loss_affichage[10:]
        plt.figure(figsize=(10, 4))
        train = list(range(len(loss_affichage_tronque)))         # [0, 1, 2, ..., 49]
        test = list(range(0, epoch, 10))   # [0, 10, 20, 30, 40]

        # 2. Les courbes
        plt.plot(train, loss_affichage_tronque, label='Train Loss', color='blue')
        plt.plot(test, loss__test_tronque, label='Test Loss', color='orange', marker='o')

        # 3. Habillage
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.title('Train vs Test Loss')
        plt.legend()
        plt.grid(True)

        # 4. Sauvegarde
        plt.savefig(os.path.join(path + "\\loss", f"loss_comparaison_tronque.png"))
        plt.close()
        
    
    

In [ ]:
train_loader, test_loader = get_data("C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P1\\projet deep L3S2\\spec")

maxim = 0
minim = np.inf
for input_, mini, maxi in train_loader:
    mini = min(mini)
    maxi = max(maxi)
    if maxi > maxim:
        maxim = maxi.item()
    if mini < minim:
        minim = mini.item()
        
print(maxim)
print(minim)

53.24430465698242
-42.518272399902344


In [8]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [9]:
class SpectrogramDataset(Dataset):
    def __init__(self, spec_path, minim, maxim):
        self.file_paths = [os.path.join(spec_path, f)
                           for f in os.listdir(spec_path) if f.endswith('.npy')]
        self.minim = minim
        self.maxim = maxim

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        spec = np.load(self.file_paths[idx])
        spec_tensor = torch.from_numpy(spec).float()
        spec_norm = (spec_tensor - self.minim) / (self.maxim - self.minim)
        return spec_norm

def get_data(spec_path, minim, maxim, batch_size=12, split_ratio=0.8):
    dataset = SpectrogramDataset(spec_path, minim, maxim)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader



In [10]:
import gc

In [ ]:
def train_model_rnn(data_loader,test_loader, model, criterion, optimizer, nepochs, device, minim ,maxim, chunk_len=200, scheduler=None):


    loss_affichage = []
    max = np.inf
    loss_test = []
    recon_loss_plot = []
    kl_loss_plot = []

    for epoch in range(nepochs):
        train_loss = 0
        model.train()
        for batch_idx, input_ in enumerate(tqdm(data_loader, desc=f"Epoch {epoch}")):
            input_ = input_.to(device)  # shape: (batch, 1025, 2000)
            optimizer.zero_grad()
            input_ = input_.unsqueeze(1)
            recon, mu, logvar = model(input_)
            
            
            loss , recon_loss, kl_loss = criterion(recon, input_, mu ,logvar)
            
            loss.backward()
            optimizer.step()

           
            recon_loss_plot.append(recon_loss.detach().cpu().item())
            kl_loss_plot.append(kl_loss.detach().cpu().item())
            
            train_loss += loss.detach().cpu().item()
            
            if max > loss.item():
                max = loss.item() 
                torch.save(model.state_dict(), "C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P1\\projet deep L3S2\\tentative_generation\\modele\\best_modele.pth")
            del input_, recon, mu, logvar, loss, recon_loss, kl_loss
            torch.cuda.empty_cache()
            gc.collect()
        avg_loss = train_loss / len(data_loader)
        loss_affichage.append(avg_loss)	
        print(f"Epoch {epoch}: Training loss = {avg_loss:.6f}")
        
        if scheduler is not None:
            scheduler.step(avg_loss)
        
        if epoch % 10 == 0:
            test_loss = 0
            model.eval()
            
            with torch.no_grad():
                for batch_idx, input_ in enumerate(test_loader):
                    input_ = input_.to(device)
                    input_ = input_.unsqueeze(1)
                    recon, mu, logvar = model(input_)
                    loss, recon_loss, kl_loss = criterion(recon, input_,mu , logvar)
                    test_loss += loss.detach().cpu().item()
                loss_test.append(test_loss/len(test_loader))
                print(f"Test loss = {test_loss/len(test_loader):.6f}")
            testeur(data_loader, model, device , epoch, loss_test , loss_affichage ,recon_loss_plot , kl_loss_plot,minim , maxim)  
            del input_, recon, mu, logvar, loss, recon_loss, kl_loss
            torch.cuda.empty_cache()
            gc.collect()
            


In [12]:
def vae_loss_fn(recon_x, x, mu, logvar):
    recon_loss = F.binary_cross_entropy(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kld , recon_loss, kld


In [13]:
# VAE basique pour images monochromes (ex: spectrogrammes)
class VAE(nn.Module):
    def __init__(self, input_channels=1, latent_dim=20):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 16, kernel_size=4, stride=2, padding=1),  # -> (16, 512, 1000)
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),              # -> (32, 256, 500)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),              # -> (64, 128, 250)
            nn.ReLU(),
            nn.Flatten()
        )

        # ⬇️ On calcule dynamiquement la taille de sortie du conv
        dummy_input = torch.zeros(1, input_channels, 1025, 2000)
        encoded_dim = self.encoder(dummy_input).shape[1]  # par ex. 64*128*250 = 2048000

        self.fc_mu = nn.Linear(encoded_dim, latent_dim)
        self.fc_logvar = nn.Linear(encoded_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, encoded_dim)

        # On "déflattene" ici vers (64, 128, 250) dans le decodeur
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (64, 128, 250)),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # -> (32, 256, 500)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),  # -> (16, 512, 1000)
            nn.ReLU(),
            nn.ConvTranspose2d(16, input_channels, kernel_size=4, stride=2, padding=1),  # -> (1, 1025, 2000)
            nn.Sigmoid()
        )


    def encode(self, x):
       
        x_encoded = self.encoder(x)
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        x_decoded = self.fc_decode(z)
        x_decoded = self.decoder(x_decoded)
        return x_decoded

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_reconstructed = self.decode(z)
        
        x_reconstructed = F.interpolate(x_reconstructed, size=(1025, 2000), mode='bilinear', align_corners=False)
        return x_reconstructed, mu, logvar

    

In [ ]:

#criterion = auraloss.freq.MultiResolutionSTFTLoss() # a tester e profondeur
#criterion = nn.MSELoss()  #avant
from torch.optim.lr_scheduler import ReduceLROnPlateau


criterion = vae_loss_fn 
model = VAE().to(device) # 0.0005 bien pour tout_spec et RNN vae

"""model = MusicRNNVAE(input_dim=1025, hidden_dim=1024, latent_dim=128, num_layers=2).to(device)"""
optimizer = torch.optim.Adam(model.parameters(), lr=0.0006) # 0.0005 bien pour tout_spec et RNN vae

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=4, verbose=True)



train_loader, test_loader = get_data("C:\\Users\\gabri\\Desktop\\dauphine\\L3\\S2\\Deep_learning\\Parties\\P1\\projet deep L3S2\\spec", minim, maxim, batch_size=12, split_ratio=0.8)

train_model_rnn(train_loader, test_loader, model, criterion, optimizer, minim = minim , maxim = maxim, nepochs=31, device=device, scheduler=scheduler)


c:\Users\gabri\anaconda3\envs\gab\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 0:  64%|██████▍   | 9/14 [00:12<00:06,  1.33s/it]


RuntimeError: CUDA error: device-side assert triggered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
